In [ ]:
import os, io, json, time, random, glob
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from datasets import load_from_disk
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Attribution Device:", device)

# 1. Resolve Data Paths
sub_paths = glob.glob("/kaggle/input/**/defactify_subset", recursive=True)
assert sub_paths, "Attach data-prep notebook containing defactify_subset!"
SAVED_ROOT = os.path.dirname(sub_paths[0])
SPLITS_DIR = f"{SAVED_ROOT}/splits"
defactify = load_from_disk(sub_paths[0])

# Locate BigGAN directory
biggan_sub = f"{SAVED_ROOT}/biggan_subset"
if os.path.exists(biggan_sub):
    BIGGAN_DIR = biggan_sub
else:
    matches = glob.glob("/kaggle/input/**/imagenet_ai_0419_biggan", recursive=True)
    BIGGAN_DIR = matches[0] if matches else SAVED_ROOT

OUT = "/kaggle/working/AI_Image_Generalization"
os.makedirs(f"{OUT}/checkpoints", exist_ok=True)
os.makedirs(f"{OUT}/results", exist_ok=True)

# 2. Attribution Dataset Class
IMG_COL = "Image"
class AttributionDataset(Dataset):
    def __init__(self, csv_path, hf_data, local_root, transform=None):
        self.rows = pd.read_csv(csv_path).to_dict("records")
        self.hf, self.root, self.tf = hf_data, local_root, transform
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        if r["storage"] == "hf":
            img = self.hf[r["hf_split"]][int(r["hf_index"])][IMG_COL]
        else:
            img = Image.open(os.path.join(self.root, r["image_path"]))
        img = img.convert("RGB")
        if self.tf: img = self.tf(img)
        return img, int(r["generator_id"])

class JPEGRecompress:
    def __init__(self, q_min=65, q_max=95): self.q_min, self.q_max = q_min, q_max
    def __call__(self, img):
        buf = io.BytesIO()
        img.save(buf, "JPEG", quality=random.randint(self.q_min, self.q_max))
        buf.seek(0)
        return Image.open(buf).convert("RGB")

norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_tf = transforms.Compose([
    transforms.RandomCrop(128, pad_if_needed=True),
    transforms.RandomHorizontalFlip(),
    JPEGRecompress(65, 95),
    transforms.ToTensor(), norm
])
eval_tf = transforms.Compose([
    transforms.CenterCrop(128),
    JPEGRecompress(90, 90),
    transforms.ToTensor(), norm
])

# 3. Model Architecture (6-class attribution)
def build_attribution_model(num_classes=6):
    m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

# 4. Loaders & Optimization
attr_dir = f"{SPLITS_DIR}/attribution"
tr_ds = AttributionDataset(f"{attr_dir}/train.csv", defactify, BIGGAN_DIR, transform=train_tf)
va_ds = AttributionDataset(f"{attr_dir}/val.csv", defactify, BIGGAN_DIR, transform=eval_tf)
te_ds = AttributionDataset(f"{attr_dir}/test.csv", defactify, BIGGAN_DIR, transform=eval_tf)

tr_loader = DataLoader(tr_ds, batch_size=128, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
va_loader = DataLoader(va_ds, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)
te_loader = DataLoader(te_ds, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)

model = build_attribution_model(num_classes=6).to(device)
crit = nn.CrossEntropyLoss()
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=5)
scaler = torch.amp.GradScaler("cuda")

best_acc = 0.0
best_path = f"{OUT}/checkpoints/attribution_resnet50_best.pt"

# Training Loop
for ep in range(5):
    model.train()
    tl, n, t0 = 0.0, 0, time.time()
    for x, y in tr_loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16):
            loss = crit(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        tl += loss.item() * len(y); n += len(y)
    sched.step()

    # Validation
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x, y in va_loader:
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(x.to(device))
            preds.extend(logits.argmax(1).cpu().numpy())
            targets.extend(y.numpy())
    val_acc = accuracy_score(targets, preds)
    print(f"[Attribution] Epoch {ep} loss={tl/n:.4f} val_acc={val_acc:.4f} ({time.time()-t0:.0f}s)")
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), best_path)

# 5. Test Evaluation
model.load_state_dict(torch.load(best_path, map_location=device, weights_only=False))
model.eval()
test_preds, test_targets = [], []
with torch.no_grad():
    for x, y in te_loader:
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(x.to(device))
        test_preds.extend(logits.argmax(1).cpu().numpy())
        test_targets.extend(y.numpy())

gen_labels = ['BigGAN', 'DALL_E_3', 'Midjourney_v6', 'Stable_Diffusion_2_1', 'Stable_Diffusion_3', 'Stable_Diffusion_XL']
prec, rec, f1, _ = precision_recall_fscore_support(test_targets, test_preds, average='macro')
overall_acc = accuracy_score(test_targets, test_preds)

print(f"\n--- Generator Attribution Results ---")
print(f"Overall Accuracy: {overall_acc:.4f}")
print(f"Macro Precision:  {prec:.4f}")
print(f"Macro Recall:     {rec:.4f}")
print(f"Macro F1-Score:   {f1:.4f}")

# 6. Confusion Matrix Plot
cm = confusion_matrix(test_targets, test_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=gen_labels, yticklabels=gen_labels)
plt.title("Generator Attribution Confusion Matrix", fontsize=14)
plt.xlabel("Predicted Generator")
plt.ylabel("True Generator")
plt.tight_layout()
plt.savefig(f"{OUT}/results/attribution_confusion_matrix.png", dpi=300)
plt.show()

# Save numeric summary
attr_summary = {
    "overall_accuracy": float(overall_acc),
    "macro_precision": float(prec),
    "macro_recall": float(rec),
    "macro_f1": float(f1),
    "confusion_matrix": cm.tolist()
}
json.dump(attr_summary, open(f"{OUT}/results/attribution_metrics.json", "w"), indent=2)

In [ ]:
import os, io, json, time, random, glob
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from datasets import load_from_disk
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("ConvNeXt Benchmark Device:", device)

# 1. Resolve Data Paths
sub_paths = glob.glob("/kaggle/input/**/defactify_subset", recursive=True)
assert sub_paths, "defactify_subset not found!"
SAVED_ROOT = os.path.dirname(sub_paths[0])
SPLITS_DIR = f"{SAVED_ROOT}/splits"
defactify = load_from_disk(sub_paths[0])

biggan_sub = f"{SAVED_ROOT}/biggan_subset"
if os.path.exists(biggan_sub):
    BIGGAN_DIR = biggan_sub
else:
    matches = glob.glob("/kaggle/input/**/imagenet_ai_0419_biggan", recursive=True)
    BIGGAN_DIR = matches[0] if matches else SAVED_ROOT

OUT = "/kaggle/working/AI_Image_Generalization"
os.makedirs(f"{OUT}/convnext_results", exist_ok=True)

# 2. Dataset and Augmentation Setup
IMG_COL = "Image"
class UnifiedImageDataset(Dataset):
    def __init__(self, csv_path, hf_data, local_root, transform=None, label_col="binary_label"):
        self.rows = pd.read_csv(csv_path).to_dict("records")
        self.hf, self.root, self.tf, self.label_col = hf_data, local_root, transform, label_col
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        if r["storage"] == "hf":
            img = self.hf[r["hf_split"]][int(r["hf_index"])][IMG_COL]
        else:
            img = Image.open(os.path.join(self.root, r["image_path"]))
        img = img.convert("RGB")
        if self.tf: img = self.tf(img)
        return img, int(r[self.label_col])

class JPEGRecompress:
    def __init__(self, q_min=65, q_max=95): self.q_min, self.q_max = q_min, q_max
    def __call__(self, img):
        buf = io.BytesIO()
        img.save(buf, "JPEG", quality=random.randint(self.q_min, self.q_max))
        buf.seek(0)
        return Image.open(buf).convert("RGB")

norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_tf = transforms.Compose([
    transforms.RandomCrop(128, pad_if_needed=True),
    transforms.RandomHorizontalFlip(),
    JPEGRecompress(65, 95),
    transforms.ToTensor(), norm
])
eval_tf = transforms.Compose([
    transforms.CenterCrop(128),
    JPEGRecompress(90, 90),
    transforms.ToTensor(), norm
])

def make_loader(csv, tf, shuffle):
    ds = UnifiedImageDataset(csv, defactify, BIGGAN_DIR, transform=tf)
    return DataLoader(ds, batch_size=128, shuffle=shuffle, num_workers=4, pin_memory=True, drop_last=shuffle)

# 3. Model Architecture
def build_convnext(num_classes=2):
    m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
    m.classifier[2] = nn.Linear(m.classifier[2].in_features, num_classes)
    return m

@torch.no_grad()
def predict(model, loader):
    model.eval(); out = []
    for x, _ in loader:
        with torch.autocast("cuda", dtype=torch.float16):
            lo = model(x.to(device, non_blocking=True))
        out.append(torch.softmax(lo.float(), 1)[:, 1].cpu())
    return torch.cat(out).numpy()

# 4. Training Function
def train_convnext_run(exp, epochs=5, lr=1e-4):
    tag = f"convnext_{exp}"
    print(f"\n--- Training {tag} ---")
    d = f"{SPLITS_DIR}/{exp}"
    tr_df = pd.read_csv(f"{d}/train.csv")
    va_df = pd.read_csv(f"{d}/val.csv")
    te_df = pd.read_csv(f"{d}/test.csv")

    tr_loader = make_loader(f"{d}/train.csv", train_tf, True)
    va_loader = make_loader(f"{d}/val.csv", eval_tf, False)
    te_loader = make_loader(f"{d}/test.csv", eval_tf, False)

    c = tr_df.binary_label.value_counts()
    w = torch.tensor([len(tr_df) / (2 * c[0]), len(tr_df) / (2 * c[1])], dtype=torch.float32, device=device)
    crit = nn.CrossEntropyLoss(weight=w)

    model = build_convnext(2).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler("cuda")

    best_auc = -1.0
    best_path = f"{OUT}/checkpoints/{tag}_best.pt"

    for ep in range(epochs):
        model.train()
        t0, tl, n = time.time(), 0.0, 0
        for x, y in tr_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = crit(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            tl += loss.item() * len(y); n += len(y)
        sched.step()

        val_preds = predict(model, va_loader)
        auc = roc_auc_score(va_df.binary_label, val_preds)
        print(f"[{tag}] ep{ep}  loss={tl/n:.4f}  val_auc={auc:.4f}  ({time.time()-t0:.0f}s)")

        if auc > best_auc:
            best_auc = auc
            torch.save(model.state_dict(), best_path)

    model.load_state_dict(torch.load(best_path, map_location=device))
    test_preds = predict(model, te_loader)
    test_auc = roc_auc_score(te_df.binary_label, test_preds)
    bal_acc = balanced_accuracy_score(te_df.binary_label, test_preds > 0.5)

    res = {"exp": exp, "architecture": "convnext_tiny", "val_auc": float(best_auc), "test_auc": float(test_auc), "bal_acc": float(bal_acc)}
    json.dump(res, open(f"{OUT}/convnext_results/{tag}.json", "w"), indent=2)
    print(f"TEST {tag}: Test AUC={test_auc:.4f}  Bal Acc={bal_acc:.4f}")
    return res

# 5. Run Selected Experiments
benchmark_exps = ["seen_all", "gan_to_diffusion", "diffusion_to_gan", "unseen_Midjourney_v6"]
convnext_summary = []
for exp in benchmark_exps:
    convnext_summary.append(train_convnext_run(exp, epochs=5))

df_compare = pd.DataFrame(convnext_summary)
display(df_compare)

In [ ]:
import os
import glob
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Base Paths Setup
OUT = "/kaggle/working/AI_Image_Generalization"
REPORT_DIR = os.path.join(OUT, "final_report")
os.makedirs(REPORT_DIR, exist_ok=True)

# -------------------------------------------------------------
# STEP 1: DYNAMICALLY READ RESNET-50 AND CONVNEXT RESULTS
# -------------------------------------------------------------

# Search for result directories across /kaggle/input and /kaggle/working
resnet_results = glob.glob("/kaggle/input/**/AI_Image_Generalization/results/*.json", recursive=True) + \
                 glob.glob("/kaggle/working/**/AI_Image_Generalization/results/*.json", recursive=True)

convnext_results = glob.glob("/kaggle/working/**/convnext_results/*.json", recursive=True) + \
                   glob.glob("/kaggle/input/**/convnext_results/*.json", recursive=True)

# Parse ResNet-50 results
resnet_dict = {}
for file_path in resnet_results:
    tag = os.path.splitext(os.path.basename(file_path))[0]
    try:
        with open(file_path, "r") as f:
            data = json.load(f)
            resnet_dict[tag] = {
                "test_auc": float(data.get("auc", np.nan)),
                "bal_acc": float(data.get("bal_acc", np.nan))
            }
    except Exception:
        continue

# Parse ConvNeXt results
convnext_dict = {}
for file_path in convnext_results:
    tag = os.path.splitext(os.path.basename(file_path))[0]
    exp = tag.replace("convnext_", "")
    try:
        with open(file_path, "r") as f:
            data = json.load(f)
            convnext_dict[exp] = {
                "test_auc": float(data.get("test_auc", np.nan)),
                "bal_acc": float(data.get("bal_acc", np.nan))
            }
    except Exception:
        continue

# Benchmark structure
experiments = [
    ("In-Distribution", "seen_all"),
    ("Cross-Generation", "gan_to_diffusion"),
    ("Cross-Generation", "diffusion_to_gan"),
    ("Unseen Generator", "unseen_BigGAN"),
    ("Unseen Generator", "unseen_DALL_E_3"),
    ("Unseen Generator", "unseen_Midjourney_v6"),
    ("Unseen Generator", "unseen_Stable_Diffusion_2_1"),
    ("Unseen Generator", "unseen_Stable_Diffusion_3"),
    ("Unseen Generator", "unseen_Stable_Diffusion_XL")
]

table1_rows = []
for category, exp in experiments:
    frozen_tag = f"{exp}_frozen_A"
    finetune_tag = f"{exp}_finetune_A"
    
    r_frozen = resnet_dict.get(frozen_tag, {})
    r_finetune = resnet_dict.get(finetune_tag, {})
    c_data = convnext_dict.get(exp, {})
    
    table1_rows.append({
        "Category": category,
        "Experiment": exp,
        "ResNet50_Frozen_AUC": r_frozen.get("test_auc", np.nan),
        "ResNet50_Finetuned_AUC": r_finetune.get("test_auc", np.nan),
        "ConvNeXt_Tiny_AUC": c_data.get("test_auc", np.nan),
        "ResNet50_Bal_Acc": r_finetune.get("bal_acc", np.nan),
        "ConvNeXt_Bal_Acc": c_data.get("bal_acc", np.nan)
    })

df_table1 = pd.DataFrame(table1_rows)
df_table1.to_csv(os.path.join(REPORT_DIR, "table1_binary_detection_benchmark.csv"), index=False)

print("="*80)
print("TABLE 1: BINARY BENCHMARK (LOADED FROM RESULT JSON FILES)")
print("="*80)
display(df_table1.round(4).fillna("-"))


# -------------------------------------------------------------
# STEP 2: LOAD ATTRIBUTION EVALUATION RESULTS
# -------------------------------------------------------------

attr_files = glob.glob("/kaggle/working/**/attribution_metrics.json", recursive=True) + \
             glob.glob("/kaggle/input/**/attribution_metrics.json", recursive=True)

gen_labels = ['BigGAN', 'DALL_E_3', 'Midjourney_v6', 'Stable_Diffusion_2_1', 'Stable_Diffusion_3', 'Stable_Diffusion_XL']

if attr_files:
    with open(attr_files[0], "r") as f:
        attr_data = json.load(f)
    
    cm_matrix = np.array(attr_data["confusion_matrix"])
    recalls = (np.diag(cm_matrix) / cm_matrix.sum(axis=1)) * 100

    df_cm = pd.DataFrame(cm_matrix, index=gen_labels, columns=gen_labels)
    df_cm["Class_Recall(%)"] = recalls.round(2)
    df_cm.to_csv(os.path.join(REPORT_DIR, "table2_generator_attribution_matrix.csv"))

    print("\n" + "="*80)
    print("TABLE 2: ATTRIBUTION RESULTS (FROM attribution_metrics.json)")
    print("="*80)
    print(f"Overall Accuracy: {attr_data.get('overall_accuracy', 0):.4f}")
    print(f"Macro Precision : {attr_data.get('macro_precision', 0):.4f}")
    print(f"Macro Recall    : {attr_data.get('macro_recall', 0):.4f}")
    print(f"Macro F1-Score  : {attr_data.get('macro_f1', 0):.4f}\n")
    display(df_cm)

    # Plot Confusion Matrix
    plt.figure(figsize=(9, 7))
    sns.heatmap(
        df_cm[gen_labels], 
        annot=True, 
        fmt="d", 
        cmap="Blues", 
        xticklabels=gen_labels,
        yticklabels=gen_labels,
        linewidths=0.5
    )
    plt.title(f"Generator Attribution Confusion Matrix (ResNet-50)\nAccuracy: {attr_data.get('overall_accuracy', 0):.2%} | F1: {attr_data.get('macro_f1', 0):.4f}", fontsize=13)
    plt.xlabel("Predicted Generator")
    plt.ylabel("True Generator")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(REPORT_DIR, "figure_attribution_confusion_matrix.png"), dpi=300)
    plt.show()
else:
    print("\nNo attribution_metrics.json found. Make sure Step 1 attribution script finished.")